# Agua CDMX PCA

Este notebook se centra exclusivamente en el archivo `consumo_agua_historico_2019.csv`.

La meta de esta version es reconstruir una tabla analitica a nivel colonia, dejando el consumo total general y su descomposicion por `indice_des`.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')


In [ ]:
base_path = Path('.')
agua_path = base_path / 'consumo_agua' / 'consumo_agua_historico_2019.csv'
output_path = base_path / 'consumo_agua' / 'tabla_colonia_consumo.csv'

agua_path


## 1. Carga inicial


In [ ]:
df = pd.read_csv(agua_path)
df['fecha_referencia'] = pd.to_datetime(df['fecha_referencia'])

print(f'Filas: {df.shape[0]:,}')
print(f'Columnas: {df.shape[1]:,}')

df.head()


## 2. Eliminacion de filas repetidas

Antes de agregar o transformar, quitamos filas exactamente repetidas.


In [ ]:
df_limpio = df.drop_duplicates().copy()

print(f'Filas antes: {len(df):,}')
print(f'Filas despues: {len(df_limpio):,}')
print(f'Filas eliminadas: {len(df) - len(df_limpio):,}')


## 3. Eliminacion de variables no prioritarias

Por ahora conservamos solo las variables necesarias para trabajar con `consumo_total` y con la composicion por `indice_des`.


In [ ]:
columnas_a_eliminar = [
    'consumo_total_mixto',
    'consumo_prom_dom',
    'consumo_total_dom',
    'consumo_prom_mixto',
    'consumo_prom',
    'consumo_prom_no_dom',
    'consumo_total_no_dom',
    'latitud',
    'longitud'
]

df_total = df_limpio.drop(columns=columnas_a_eliminar).copy()

df_total.head()


## 4. Codificacion ordinal de indice_des

Tratamos `indice_des` como una variable ordinal para conservar su orden en una forma numerica.


In [ ]:
mapa_indice_des = {
    'POPULAR': 1,
    'BAJO': 2,
    'MEDIO': 3,
    'ALTO': 4
}

df_total['indice_des_num'] = df_total['indice_des'].map(mapa_indice_des)

df_total[['indice_des', 'indice_des_num']].drop_duplicates().sort_values('indice_des_num')


## 5. Colapso de bimestre

Si una misma combinacion de `alcaldia + colonia + indice_des` aparece en varios bimestres, sumamos `consumo_total` para dejar una sola fila por combinacion.


In [ ]:
df_colapsado = (
    df_total.groupby(['alcaldia', 'colonia', 'indice_des', 'indice_des_num'], as_index=False)
    .agg(consumo_total=('consumo_total', 'sum'))
)

df_colapsado.head()


## 6. Tabla final por colonia

Ahora construimos una tabla con una sola fila por `alcaldia + colonia`. Para cada nivel de `indice_des` se crea una columna de consumo y otra de porcentaje respecto al `consumo_total` de la colonia.

Nota: se usa `alcaldia + colonia` como llave, porque hay nombres de colonia repetidos entre alcaldias.


In [ ]:
tabla_colonia = (
    df_colapsado.pivot_table(
        index=['alcaldia', 'colonia'],
        columns='indice_des',
        values='consumo_total',
        aggfunc='sum',
        fill_value=0
    )
    .rename(columns={
        'POPULAR': 'consumo_popular',
        'BAJO': 'consumo_bajo',
        'MEDIO': 'consumo_medio',
        'ALTO': 'consumo_alto'
    })
    .reset_index()
)

for col in ['consumo_popular', 'consumo_bajo', 'consumo_medio', 'consumo_alto']:
    if col not in tabla_colonia.columns:
        tabla_colonia[col] = 0

tabla_colonia['consumo_total'] = (
    tabla_colonia['consumo_popular']
    + tabla_colonia['consumo_bajo']
    + tabla_colonia['consumo_medio']
    + tabla_colonia['consumo_alto']
)

tabla_colonia['perc_popular'] = np.where(tabla_colonia['consumo_total'] > 0, tabla_colonia['consumo_popular'] / tabla_colonia['consumo_total'], 0)
tabla_colonia['perc_bajo'] = np.where(tabla_colonia['consumo_total'] > 0, tabla_colonia['consumo_bajo'] / tabla_colonia['consumo_total'], 0)
tabla_colonia['perc_medio'] = np.where(tabla_colonia['consumo_total'] > 0, tabla_colonia['consumo_medio'] / tabla_colonia['consumo_total'], 0)
tabla_colonia['perc_alto'] = np.where(tabla_colonia['consumo_total'] > 0, tabla_colonia['consumo_alto'] / tabla_colonia['consumo_total'], 0)

tabla_colonia = tabla_colonia[[
    'alcaldia',
    'colonia',
    'consumo_total',
    'consumo_popular',
    'consumo_bajo',
    'consumo_medio',
    'consumo_alto',
    'perc_popular',
    'perc_bajo',
    'perc_medio',
    'perc_alto'
]].sort_values(['alcaldia', 'colonia']).reset_index(drop=True)

tabla_colonia.head()


In [ ]:
tabla_colonia.to_csv(output_path, index=False)
print(f'CSV guardado en: {output_path}')


In [ ]:
tabla_colonia.describe(include='all')


## Resultado

La tabla final para seguir trabajando es `tabla_colonia`, y se guarda como CSV en `consumo_agua/tabla_colonia_consumo.csv`.
